# Triple Barrier Method — Exercise 3.5

Split from the original AFML 3.2 notebook. The ML follow-up (meta-labeling, Exercise 3.5b) lives in `AFML 3.2.2 - Meta-Labeling with sklearn.ipynb`.

Most of the functions below can be found under research/Labels.

Contact: boyboi86@gmail.com


In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

p = print

dollar = pd.read_csv('../sample-data/dollar_bars.csv', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])


In [ ]:
# fraction form keeps the original pipeline (dollar) intact; moment form goes to a parallel frame
dollar['ewm'], dollar['upper'], dollar['lower'], dollar['std'] = rs.bband_frac(dollar)
dollar_m = dollar.copy()
dollar_m['ewm'], dollar_m['upper'], dollar_m['lower'], dollar_m['std'] = rs.bband_std(dollar_m)


In [ ]:
dollar['side'] = np.nan

upper = dollar[dollar['upper'] < dollar['close']] # short signal
lower = dollar[dollar['lower'] > dollar['close']] # long signal

p("Num of times upper limit touched: {0}\nNum of times lower limit touched: {1}"
  .format(len(upper), 
          len(lower)))

# Recall white test as a benchmark and until this stage we filtered all those which did not meet min return
dollar = rs.side_pick(dollar)
dollar.dropna(inplace= True)
dollar['side'].value_counts()


In [ ]:
copy_dollar = dollar.copy() # make a back copy used by the ML exercise (AFML 3.2.2)

copy_dollar # up till this point the below dataframe should look like this, before tri_bar func. This is our primary model.


In [ ]:
d_vol = rs.vol(dollar['close'], span0 = 50)


In [ ]:
# d_vol is a return (~0.55%); cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = d_vol.mean() * dollar['close'].mean())

events


In [ ]:
vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

vb # Show some example output


In [ ]:
tb = rs.tri_barrier(data = dollar['close'], 
                events = events, 
                trgt = d_vol, 
                min_req = 0.002, 
                num_threads = 3, 
                ptSl = [0,2], # change ptSl into [0,2]
                t1 = vb, 
                side = dollar['side'])

tb # Show some example


In [ ]:
m_label = rs.meta_label(data = dollar['close'],
                      events = tb,
                      drop = False)

m_label # Show some example


In [ ]:
m_label['bin'].value_counts(normalize=True)

# Here is a quick look at our 'bin' values.
# Slight imbalanced sample, but not much harm
# 51.95% of the sample based on parameter touched vertical barrier first
